In [111]:
import pandas as pd
import requests
import time



In [112]:
movies = pd.read_csv(
    "MovieLens 1M/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names= ['MovieID','Title','Genres']
)

ratings = pd.read_csv(
    "MovieLens 1M/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names = ['UserID','MovieID','Rating','Timestamp']
)
links = pd.read_csv('data/movie lense 20m/link.csv')
links['tmdbId'] = links['tmdbId'].astype('Int64')


In [113]:
movies.shape

(3883, 3)

In [114]:
movies_with_ratings = ratings['MovieID'].unique()


len(movies_with_ratings)

3706

Taking only the movies, we have ratings for

In [115]:
movies = movies[movies['MovieID'].isin(movies_with_ratings)]

movies.shape

(3706, 3)

Merging movies and link.csv to get imdb/tmdb id for our movies

In [116]:
movies = movies.merge(links, left_on="MovieID",right_on='movieId', how = 'left')
display(movies[movies['imdbId'].isna()].shape)
display(movies[movies['imdbId'].isna()])

(21, 6)

,MovieID,Title,Genres,movieId,imdbId,tmdbId
543,557,Mamma Roma (1962),Drama,NaN,NaN,<NA>
564,578,"Hour of the Pig, The (1993)",Drama|Mystery,NaN,NaN,<NA>
647,669,Aparajito (1956),Drama,NaN,NaN,<NA>
765,811,"Bewegte Mann, Der (1994)",Comedy,NaN,NaN,<NA>
807,863,Celestial Clockwork (1994),Comedy,NaN,NaN,<NA>
917,978,"Blue Angel, The (Blaue Engel, Der) (1930)",Drama,NaN,NaN,<NA>
1115,1205,"Transformers: The Movie, The (1986)",Action|Animation|Children's|Sci-Fi|Thriller|War,NaN,NaN,<NA>
1202,1294,M*A*S*H (1970),Comedy|War,NaN,NaN,<NA>
1264,1362,"Garden of Finzi-Contini, The (Giardino dei Fin...",Drama,NaN,NaN,<NA>
1378,1494,"Sixth Man, The (1997)",Comedy,NaN,NaN,<NA>


We were unable to find IMDb IDs for these 21 movies in any of the MovieLens datasets (100K, 1M, or 20M). The 20M dataset was specifically downloaded to obtain complete movie ID mappings, but some entries are still missing. Note that the model itself was trained on the 1M dataset.

I will set the missing IMDb IDs manually by visiting each movie’s IMDb page.

In [117]:
movies.isna().sum()

MovieID     0
Title       0
Genres      0
movieId    21
imdbId     21
tmdbId     40
dtype: int64

In [118]:
movies = movies.drop(columns = ['movieId','tmdbId'])

In [119]:
display(movies[movies['imdbId'].isna()])


,MovieID,Title,Genres,imdbId
543,557,Mamma Roma (1962),Drama,NaN
564,578,"Hour of the Pig, The (1993)",Drama|Mystery,NaN
647,669,Aparajito (1956),Drama,NaN
765,811,"Bewegte Mann, Der (1994)",Comedy,NaN
807,863,Celestial Clockwork (1994),Comedy,NaN
917,978,"Blue Angel, The (Blaue Engel, Der) (1930)",Drama,NaN
1115,1205,"Transformers: The Movie, The (1986)",Action|Animation|Children's|Sci-Fi|Thriller|War,NaN
1202,1294,M*A*S*H (1970),Comedy|War,NaN
1264,1362,"Garden of Finzi-Contini, The (Giardino dei Fin...",Drama,NaN
1378,1494,"Sixth Man, The (1997)",Comedy,NaN


In [120]:
movies.loc[movies['MovieID'] == 557, 'imdbId'] = 56215
movies.loc[movies['MovieID'] == 578, 'imdbId'] = 107146
movies.loc[movies['MovieID'] == 669, 'imdbId'] = 48956
movies.loc[movies['MovieID'] == 811, 'imdbId'] = 109255
movies.loc[movies['MovieID'] == 863, 'imdbId'] = 107537
movies.loc[movies['MovieID'] == 978, 'imdbId'] = 20697
movies.loc[movies['MovieID'] == 1205, 'imdbId'] = 92106
movies.loc[movies['MovieID'] == 1294, 'imdbId'] = 66026
movies.loc[movies['MovieID'] == 1362, 'imdbId'] = 65777
movies.loc[movies['MovieID'] == 1494, 'imdbId'] = 120142
movies.loc[movies['MovieID'] == 1741, 'imdbId'] = 112619
movies.loc[movies['MovieID'] == 1758, 'imdbId'] = 118892
movies.loc[movies['MovieID'] == 1868, 'imdbId'] = 117959
movies.loc[movies['MovieID'] == 2645, 'imdbId'] = 51554
movies.loc[movies['MovieID'] == 3027, 'imdbId'] = 96117
movies.loc[movies['MovieID'] == 3366, 'imdbId'] = 65207
movies.loc[movies['MovieID'] == 3416, 'imdbId'] = 57427
movies.loc[movies['MovieID'] == 3482, 'imdbId'] = 188160
movies.loc[movies['MovieID'] == 3532, 'imdbId'] = 22599
movies.loc[movies['MovieID'] == 3842, 'imdbId'] = 82700
movies.loc[movies['MovieID'] == 3935, 'imdbId'] = 71276

In [121]:
display(movies[movies['imdbId'].isna()])


,MovieID,Title,Genres,imdbId


finnaly we have imdb movies for

In [1]:
API_KEY = ""

Getting movie metadata using tmbd api

## Claude Code for downloading movie metadata using tmdb api

In [35]:
import pandas as pd
import requests
import time
import json
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

BASE_URL        = "https://api.themoviedb.org/3"
POSTER_BASE_URL = "https://image.tmdb.org/t/p/w500"
CHECKPOINT_FILE = "tmdb_imdb_checkpoint.json"
OUTPUT_FILE     = "movies_tmdb_enriched.csv"
WORKERS         = 5


def make_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    return session

SESSION = make_session()


def get(url, params):
    for attempt in range(1, 6):
        try:
            r = SESSION.get(url, params=params, timeout=10)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 10))
                print(f"    Rate-limited — waiting {wait}s …")
                time.sleep(wait)
                continue
            return r
        except requests.exceptions.ConnectionError:
            wait = 2 ** attempt
            print(f"    Connection error, retrying in {wait}s …")
            time.sleep(wait)
    return None


def fetch_all(raw_imdb_id):
    
    # converting  raw id to imdb api required format
    imdb_id = f"tt{int(raw_imdb_id):07d}" if not str(raw_imdb_id).startswith("tt") else raw_imdb_id
    
    # Step 1: imdb_id → tmdb_id
    r = get(f"{BASE_URL}/find/{imdb_id}", {
        "api_key": API_KEY,
        "external_source": "imdb_id"
    })
    if r is None or r.status_code != 200:
        return None
    movie_results = r.json().get("movie_results", [])
    if not movie_results:
        return None
    tmdb_id = movie_results[0]["id"]

    # Step 2: tmdb_id → all details in one call
    r = get(f"{BASE_URL}/movie/{tmdb_id}", {
        "api_key": API_KEY,
        "append_to_response": "keywords,credits"
    })
    if r is None or r.status_code != 200:
        return None

    d = r.json()
    poster_path = d.get("poster_path")

    return {
        "imdbId":               raw_imdb_id,
        "tmdbId":               tmdb_id,
        "title":                d.get("title"),
        "overview":             d.get("overview"),
        "original_language":    d.get("original_language"),
        "genres":               [g["name"] for g in d.get("genres", [])],
        "production_companies": [p["name"] for p in d.get("production_companies", [])],
        "keywords":             [k["name"] for k in d.get("keywords", {}).get("keywords", [])],
        "cast":                 [c["name"] for c in d.get("credits", {}).get("cast", [])[:10]],
        "crew":                 [c["name"] for c in d.get("credits", {}).get("crew", []) if c.get("job") == "Director"],
        "poster_url":           f"{POSTER_BASE_URL}{poster_path}" if poster_path else None,
    }


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            data = json.load(f)
        print(f"Resuming from checkpoint: {len(data['results'])} movies already done.")
        return set(data["done_ids"]), data["results"]
    return set(), []

def save_checkpoint(done_ids, results):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"done_ids": list(done_ids), "results": results}, f)


# ── Main ──────────────────────────────────────────────────────────────────────
# Your CSV must have a column named 'imdbId' with values like tt0468569


done_ids, results = load_checkpoint()
remaining = [row["imdbId"] for _, row in movies.iterrows() if row["imdbId"] not in done_ids]
skipped   = 0
lock      = Lock()

print(f"{len(remaining)} movies to fetch with {WORKERS} parallel workers …\n")

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(fetch_all, iid): iid for iid in remaining}

    for i, future in enumerate(as_completed(futures), 1):
        imdb_id = futures[future]
        data    = future.result()

        with lock:
            if data:
                results.append(data)
                done_ids.add(imdb_id)
                print(f"[{i}/{len(remaining)}] {imdb_id} ✓  →  {data['title']}")
            else:
                skipped += 1
                done_ids.add(imdb_id)
                print(f"[{i}/{len(remaining)}] {imdb_id} ✗ skipped")

            if i % 50 == 0:
                save_checkpoint(done_ids, results)

save_checkpoint(done_ids, results)
df_result = pd.DataFrame(results)
df_result.to_csv(OUTPUT_FILE, index=False)
os.remove(CHECKPOINT_FILE)

print(f"\n✅ Done! {len(df_result)} movies saved to '{OUTPUT_FILE}' ({skipped} skipped).")

Resuming from checkpoint: 848 movies already done.
2856 movies to fetch with 5 parallel workers …

[1/2856] 47437 ✓  →  Sabrina
[2/2856] 46250 ✓  →  Roman Holiday
[3/2856] 31580 ✓  →  The Little Princess
[4/2856] 58385 ✓  →  My Fair Lady
[5/2856] 37059 ✓  →  Meet Me in St. Louis
[6/2856] 32138 ✓  →  The Wizard of Oz
[7/2856] 31381 ✓  →  Gone with the Wind
[8/2856] 84370 ✓  →  My Favorite Year
[9/2856] 43014 ✓  →  Sunset Boulevard
[10/2856] 33467 ✓  →  Citizen Kane
[11/2856] 62622 ✓  →  2001: A Space Odyssey
[12/2856] 39428 ✓  →  Golden Earrings
[13/2856] 42192 ✓  →  All About Eve
[14/2856] 32143 ✓  →  The Women
[15/2856] 32976 ✓  →  Rebecca
[16/2856] 32484 ✓  →  Foreign Correspondent
[17/2856] 38787 ✓  →  Notorious
[18/2856] 38109 ✓  →  Spellbound
[19/2856] 50105 ✓  →  An Affair to Remember
[20/2856] 48728 ✓  →  To Catch a Thief
[21/2856] 42451 ✓  →  Father of the Bride
[22/2856] 45537 ✓  →  The Band Wagon
[23/2856] 31725 ✓  →  Ninotchka
[24/2856] 50658 ✓  →  Love in the Afternoon
[25/

In [123]:
fetched_ids = set(df_result['imdbId'])
skipped_movies = movies[~movies['imdbId'].isin(fetched_ids)]
print(skipped_movies[['MovieID', 'Title', 'imdbId']])

      MovieID                                              Title    imdbId
689       720  Wallace & Gromit: The Best of Aardman Animatio...  118114.0
696       730                               Low Life, The (1994)  125877.0
1496     1630                        Lay of the Land, The (1997)  123953.0
2068     2258                              Master Ninja I (1984)   87690.0
2315     2510                             Just the Ticket (1999)  134948.0
2644     2851                                    Saturn 3 (1979)   81454.0
2787     2999                          Man of the Century (1999)  154827.0


#### Some old movies werent available on TMDB, so we will be skipping them.

Merging Movie lense and tmdb meta data df

In [124]:
movies.columns

Index(['MovieID', 'Title', 'Genres', 'imdbId'], dtype='str')

In [125]:
df_result.columns

Index(['imdbId', 'tmdbId', 'title', 'overview', 'original_language', 'genres',
       'production_companies', 'keywords', 'cast', 'crew', 'poster_url'],
      dtype='str')

**extracting release year from title**

In [126]:
movies['release_year'] = movies['Title'].apply(lambda x : x[-5:-1])

movies['release_year'] = movies['release_year'].astype('int')

movies = movies[['MovieID',	'Title',	'Genres',	'release_year', 'imdbId']]

In [127]:
movies.columns

Index(['MovieID', 'Title', 'Genres', 'release_year', 'imdbId'], dtype='str')

In [128]:
df_result.columns

Index(['imdbId', 'tmdbId', 'title', 'overview', 'original_language', 'genres',
       'production_companies', 'keywords', 'cast', 'crew', 'poster_url'],
      dtype='str')

In [129]:
df_result.shape

(3699, 11)

In [130]:
movies = movies.merge(df_result, on = 'imdbId', how = 'left')

In [131]:
movies.shape

(3712, 15)

In [132]:
movies.isna().sum()

MovieID                  0
Title                    0
Genres                   0
release_year             0
imdbId                   0
tmdbId                   7
title                    7
overview                 7
original_language        7
genres                   7
production_companies     7
keywords                 7
cast                     7
crew                     7
poster_url              15
dtype: int64

Since these are pretty old movies, tmdb did not have complete data for them

we are missing 0.4% data but that's alright

In [133]:
movies[movies['poster_url'].isna()]

,MovieID,Title,Genres,release_year,imdbId,tmdbId,title,overview,original_language,genres,production_companies,keywords,cast,crew,poster_url
36,37,Across the Sea of Time (1995),Documentary,1995,112286.0,139405.0,Across the Sea of Time,"A young Russian boy, Thomas Minton, travels to...",en,"[Adventure, History, Drama, Family]","[Sony New Technologies, Columbia Pictures, Son...",[],"[Peter Reznick, John McDonough, Avi Hoffman, M...",[Stephen Low],NaN
134,139,Target (1995),Action|Drama,1995,114618.0,124639.0,Target,A subtle yet violent commentary on feudal lords.,hi,[Drama],[],[],"[Mohan Agashe, Om Puri, Barun Chakraborty, Gul...",[Sandip Ray],NaN
390,404,Brother Minister: The Assassination of Malcolm...,Documentary,1994,109339.0,316098.0,Brother Minister: The Assassination of Malcolm X,Brother Minister reveals the mystery surroundi...,en,[Documentary],[],[],[Roscoe Lee Browne],"[Jefri Aalmuhammed, Jack Baxter]",NaN
689,720,Wallace & Gromit: The Best of Aardman Animatio...,Animation,1996,118114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
696,730,"Low Life, The (1994)",Drama,1994,125877.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
775,826,Diebinnen (1995),Drama,1995,112865.0,282919.0,Diebinnen,,de,[Drama],[],[],"[Christiane Hörbiger, Jennifer Nitsch, Lena St...",[Peter Weck],NaN
1064,1142,Get Over It (1996),Drama,1996,116403.0,606151.0,Get Over It,Steven is depressed because he's just been dum...,en,[],[],[],"[Christian Palacios Canterbury, Dave McCrea, D...",[Nick Katsapetses],NaN
1223,1316,Anna (1996),Drama,1996,115548.0,1165738.0,Anna Meintjies,South African TV Movie,af,[],[South African Broadcasting Corporation (SABC)],[],"[Marius Weyers, Sandra Prinsloo, Trix Pienaar,...",[Manie van Rensburg],NaN
1498,1630,"Lay of the Land, The (1997)",Comedy|Drama,1997,123953.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1624,1787,Paralyzing Fear: The Story of Polio in America...,Documentary,1998,145302.0,83593.0,A Paralyzing Fear: The Story of Polio in America,This is a very clear and personalized presenta...,en,[Documentary],[],[],[],[],NaN


In [134]:
movies = movies.dropna()
movies.shape

(3697, 15)

In [138]:
movies['imdbId'] = movies['imdbId'].astype(int)
movies['tmdbId'] = movies['tmdbId'].astype(int)

In [140]:
movies.head()

,MovieID,Title,Genres,release_year,imdbId,tmdbId,title,overview,original_language,genres,production_companies,keywords,cast,crew,poster_url
0,1,Toy Story (1995),Animation|Children's|Comedy,1995,114709,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",en,"[Family, Comedy, Animation, Adventure]",[Pixar],"[rescue, friendship, mission, jealousy, villai...","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter],https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1995,113497,8844,Jumanji,When siblings Judy and Peter discover an encha...,en,"[Adventure, Fantasy, Family]","[TriStar Pictures, Interscope Communications, ...","[giant insect, board game, disappearance, jung...","[Robin Williams, Kirsten Dunst, Bradley Pierce...",[Joe Johnston],https://image.tmdb.org/t/p/w500/vgpXmVaVyUL7GG...
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,113228,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,en,"[Romance, Comedy]","[Lancaster Gate, Warner Bros. Pictures]","[fishing, sequel, old man, best friend, weddin...","[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",[Howard Deutch],https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama,1995,114885,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",en,"[Comedy, Drama, Romance]",[20th Century Fox],"[based on novel or book, single mother, divorc...","[Whitney Houston, Angela Bassett, Loretta Devi...",[Forest Whitaker],https://image.tmdb.org/t/p/w500/8MprEuTY3EwkF9...
4,5,Father of the Bride Part II (1995),Comedy,1995,113041,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,en,"[Comedy, Family]","[Touchstone Pictures, Sandollar Productions]","[daughter, baby, parent child relationship, mi...","[Steve Martin, Diane Keaton, Martin Short, Kim...",[Charles Shyer],https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


In [141]:
movies.to_csv('MovieLense-TMDB-combined-data.csv')